# GDAI-VALID — 01 · Sélection humaine des images ResNet-18 V2

**Objectif :** préparer jusqu’à **120 images** (`garden`, `not_garden`, `unusable`), en conservant les sources originales. Aucune photo n’est modifiée dans les répertoires d’ingestion. Les photos du test V1 sont exclues des candidats et restent un test historique réservé.

**Important :** une image floue n’est `unusable` que si son contenu est réellement inexploitable. Les images de provenance inconnue doivent rester locales et ne doivent pas être commitées sans vérification de licence. Les dérivés floutés sont créés **après** les splits, uniquement pour l'entraînement.

In [1]:
# Dépendances : pip install pillow pandas ipywidgets scikit-learn matplotlib
import hashlib, json, csv, shutil
import pandas as pd
from PIL import Image, ImageOps, ImageFilter
from IPython.display import display, clear_output
import ipywidgets as widgets
from sklearn.model_selection import GroupShuffleSplit


In [2]:
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'validation/app').is_dir() and (p/'models').is_dir()), None)
if ROOT is None: raise RuntimeError('Run notebook from inside GardnAIze repository')
DATA = ROOT/'datasets/validation/resnet18'
DATA.mkdir(parents=True,exist_ok=True)
print('Repository:',ROOT)

Repository: /home/bsui/projects/GardnAIze


In [3]:
SOURCES = [
 ('v1_validation', ROOT/'datasets/images/garden_validation'),
 ('ingestion_images', ROOT/'ai-service/plant-ingestion/inputs/images'),
 ('ingestion_other', ROOT/'ai-service/plant-ingestion/inputs/images-other'),
]
TEST_V1 = ROOT/'datasets/images/garden_test'
EXT={'.jpg','.jpeg','.png','.webp'}
def sha(path):
 h=hashlib.sha256()
 with path.open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
rows=[]
for source,folder in SOURCES:
 if not folder.exists():print('Absent:',folder);continue
 for p in sorted(folder.rglob('*')):
  if not p.is_file() or p.suffix.lower() not in EXT:continue
  try:
   with Image.open(p) as im: im.verify()
  except Exception as e:
   print('Unreadable:',p,e);continue
  label=p.parent.name if source=='v1_validation' and p.parent.name in {'garden','not_garden','unusable'} else 'exclude'
  rows.append(dict(path=p.relative_to(ROOT).as_posix(),source=source,label=label,selected=(source=='v1_validation'),sha256=sha(p),license='',source_url='',group_id='',reviewed=False))
manifest=DATA/'candidates_v2.csv'
new=pd.DataFrame(rows)
if manifest.exists():
 old=pd.read_csv(manifest, keep_default_na=False); preserved=old.set_index('path').to_dict('index')
 for i,r in new.iterrows():
  if r.path in preserved:
   for k in ('label','selected','license','source_url','group_id','reviewed'):
    if k in preserved[r.path]:new.at[i,k]=preserved[r.path][k]
new.to_csv(manifest,index=False)
print('Candidates:',len(new),'Exact duplicate groups:',new.sha256.duplicated().sum());display(new.groupby('source').size())


Candidates: 181 Exact duplicate groups: 1


source
ingestion_images    46
ingestion_other     63
v1_validation       72
dtype: int64

## Revue interactive

Sélectionner une image, vérifier sa classe, cocher **Conserver**, renseigner sa licence et son groupe de provenance (même jardin / même série / même photo originale). Cliquer **Enregistrer et suivant**. Les décisions sont sauvegardées immédiatement dans le CSV. Les images non vérifiées peuvent être exclues. Pour les photos d’ingestion, `exclude` est la valeur initiale.

In [4]:
df=pd.read_csv(manifest, keep_default_na=False)
# Restore booleans from CSV reliably
for col in ('selected','reviewed'):
 df[col]=df[col].astype(str).str.lower().isin(['true','1','yes'])
pos=0
img=widgets.Image(format='jpeg',width=540)
info=widgets.HTML()
label=widgets.Dropdown(options=['garden','not_garden','unusable','exclude'],description='Classe')
selected=widgets.Checkbox(description='Conserver')
license_box=widgets.Text(description='Licence',placeholder='Ex. CC BY-SA 4.0 / unknown')
url_box=widgets.Text(description='URL source',placeholder='URL si connue')
group=widgets.Text(description='Groupe',placeholder='Même scène / même source')
prev=widgets.Button(description='Précédent');next_btn=widgets.Button(description='Enregistrer et suivant',button_style='success')
status=widgets.Output()
def show(i):
 global pos
 pos=max(0,min(i,len(df)-1));r=df.iloc[pos];p=ROOT/r.path
 with Image.open(p) as im:
  im=ImageOps.exif_transpose(im).convert('RGB');im.thumbnail((800,550));from io import BytesIO
  b=BytesIO();im.save(b,format='JPEG',quality=80);img.value=b.getvalue()
 info.value=f'<b>{pos+1}/{len(df)}</b> — {r.path} — {r.source}'
 label.value=r.label if r.label in label.options else 'exclude'
 selected.value=bool(r.selected);license_box.value=str(r.license);url_box.value=str(r.source_url)
 group.value=str(r.group_id) if str(r.group_id) else str(r.sha256)
def save_current():
 for k,v in [('label',label.value),('selected',selected.value),('license',license_box.value),('source_url',url_box.value),('group_id',group.value),('reviewed',True)]:df.at[pos,k]=v
 df.to_csv(manifest,index=False)
def on_next(_):save_current();show(pos+1)
def on_prev(_):save_current();show(pos-1)
next_btn.on_click(on_next);prev.on_click(on_prev)
if len(df):display(widgets.VBox([info,img,label,selected,license_box,url_box,group,widgets.HBox([prev,next_btn]),status]));show(0)
else:print('No candidate images')


## Contrôles, échantillonnage et splits\n
Le notebook sélectionne uniquement les images revues et conservées, élimine les doublons exacts, puis cherche une répartition **stratifiée par groupes** : train ~60 %, validation ~20 %, test ~20 %. Chaque classe doit être présente dans chaque ensemble.\n
\n
**72 images sélectionnées sur 120 visées : ce n'est pas bloquant** pour un essai V2, mais le dataset reste petit. La sélection n'ajoute pas automatiquement les images d'ingestion : il faut les examiner dans la cellule de revue.\n
\n
Les images du test historique V1 restent exclues de l'entraînement et de ce nouveau découpage.\n

In [5]:
TARGET={'garden':50,'not_garden':40,'unusable':30}
df=pd.read_csv(manifest, keep_default_na=False)
for col in ('selected','reviewed'):df[col]=df[col].astype(str).str.lower().isin(['true','1','yes'])
chosen=df[df.selected & df.reviewed & df.label.isin(TARGET)].copy()
chosen=chosen.drop_duplicates('sha256',keep='first')
# Prevent exact overlap with historical test
historical_hashes={sha(p) for p in TEST_V1.rglob('*') if p.is_file() and p.suffix.lower() in EXT} if TEST_V1.exists() else set()
chosen=chosen[~chosen.sha256.isin(historical_hashes)].copy()
chosen['group_id']=chosen.apply(lambda r:str(r.group_id).strip() or r.sha256,axis=1)
# Use deterministic selection by class, no automatic forced label
chosen=pd.concat([g.sort_values('sha256').head(TARGET[label]) for label,g in chosen.groupby('label')],ignore_index=True) if len(chosen) else chosen
print('Selected:',len(chosen),'target:',sum(TARGET.values()))
print(chosen.label.value_counts().to_string() if len(chosen) else 'No reviewed images yet')
if len(chosen)<sum(TARGET.values()):print('WARNING: Fewer than 120 reviewed, distinct eligible images. Review/add candidates.')
if len(chosen)<15 or chosen.group_id.nunique()<6:
 raise RuntimeError('Insufficient images/groups for a meaningful train/val/test split')

# Stratified GROUP split: keeps each scene/group in one split and seeks all 3 labels
# Five folds => approximately 60% train, 20% validation, 20% test.
from sklearn.model_selection import StratifiedGroupKFold
from itertools import combinations
best=None
for seed in range(42, 62):
 cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
 folds=[chosen.iloc[test_idx].copy() for _,test_idx in cv.split(chosen, chosen.label, chosen.group_id)]
 for vi,ti in combinations(range(5),2):
  val,test=folds[vi],folds[ti]
  train=pd.concat([folds[k] for k in range(5) if k not in (vi,ti)],ignore_index=True)
  if any(part.label.nunique()!=len(TARGET) for part in (train,val,test)):
   continue
  # Prefer balanced class distribution and approximately 60/20/20 split
  total=len(chosen)
  score=sum(abs(len(part)/total-target) for part,target in ((train,.6),(val,.2),(test,.2)))
  for part in (train,val,test):
   for cls in TARGET:
    score+=abs((part.label==cls).mean()-(chosen.label==cls).mean())*.25
  if best is None or score<best[0]:best=(score,seed,train,val,test)
if best is None:
 raise RuntimeError(
  'No valid stratified group split found. Check group_id and class counts. '
  'Each class must occur in several independent groups; review labels and groups.'
 )
_,seed,train,val,test=best
train=train.copy();val=val.copy();test=test.copy()
train['split']='train';val['split']='val';test['split']='test'
splits=pd.concat([train,val,test],ignore_index=True)
assert splits.groupby('group_id').split.nunique().max()==1
assert not splits.sha256.isin(historical_hashes).any()
assert all(splits[splits.split==sp].label.nunique()==len(TARGET) for sp in ('train','val','test'))
print('Stratified group split selected; random seed:',seed)
split_file=ROOT/'datasets/splits/garden_split_v2.csv';split_file.parent.mkdir(parents=True,exist_ok=True)
splits.to_csv(split_file,index=False)
print('Wrote:',split_file);display(pd.crosstab(splits.split,splits.label))


Selected: 72 target: 120
label
not_garden    25
unusable      24
garden        23
Stratified group split selected; random seed: 42
Wrote: /home/bsui/projects/GardnAIze/datasets/splits/garden_split_v2.csv


label,garden,not_garden,unusable
split,,,
test,4,5,5
train,14,15,14
val,5,5,5


## Floutage : uniquement après séparation des groupes

Les images originales ne sont jamais écrasées. Générer des versions floutées uniquement à partir du **train**, avec un groupe identique à l'original. **Attention :** le floutage ne doit pas transformer automatiquement toute image en `unusable`; la classe `unusable` doit correspondre à une image réellement inexploitable. Cette cellule prépare une liste de dérivés à revoir, et ne les ajoute **pas automatiquement** à l'entraînement.

In [6]:
blur_dir=DATA/'blur_candidates_train';blur_dir.mkdir(exist_ok=True,parents=True)
blur_rows=[]
for _,r in splits[(splits.split=='train') & (splits.label.isin(['garden','not_garden']))].head(15).iterrows():
 p=ROOT/r.path
 with Image.open(p) as im:
  im=ImageOps.exif_transpose(im).convert('RGB').filter(ImageFilter.GaussianBlur(radius=12))
  dest=blur_dir/(Path(r.path).stem+'_'+r.sha256[:10]+'_blur.jpg');im.save(dest,quality=88)
 blur_rows.append({'path':dest.relative_to(ROOT).as_posix(),'parent_path':r.path,'group_id':r.group_id,'proposed_label':'unusable','approved':False,'split':'train'})
blur_manifest=DATA/'blur_review_v2.csv'
if not blur_manifest.exists():pd.DataFrame(blur_rows).to_csv(blur_manifest,index=False)
print('Blur candidates:',len(blur_rows),'Review before approval:',blur_manifest)


Blur candidates: 15 Review before approval: /home/bsui/projects/GardnAIze/datasets/validation/resnet18/blur_review_v2.csv


### Export pour l'entraînement

Après revue manuelle des flous, mettre `approved=True` uniquement pour les photos réellement inutilisables. Cette cellule crée un manifeste final sans déplacer les images. **Ne pas commiter des images sans droit de redistribution**, ni les images de quarantaine.

In [7]:
blurs=pd.read_csv(blur_manifest, keep_default_na=False) if blur_manifest.exists() else pd.DataFrame()
final=splits.copy()
if len(blurs):
 approved=blurs[blurs.approved.astype(str).str.lower().isin(['true','1','yes'])].copy()
 if len(approved):
  additions=pd.DataFrame({'path':approved.path,'label':'unusable','group_id':approved.group_id,'split':'train','source':'blur_augmentation','license':'derived_from_parent','source_url':'','sha256':[sha(ROOT/p) for p in approved.path]})
  final=pd.concat([final,additions],ignore_index=True)
assert final.groupby('group_id').split.nunique().max()==1
final_file=DATA/'training_manifest_v2.csv';final.to_csv(final_file,index=False)
print('Ready:',final_file);display(pd.crosstab(final.split,final.label))


Ready: /home/bsui/projects/GardnAIze/datasets/validation/resnet18/training_manifest_v2.csv


label,garden,not_garden,unusable
split,,,
test,4,5,5
train,14,15,14
val,5,5,5
